# IMU Signal Synchronization & Inspection

Visual debugging of cross-volunteer temporal alignment for individual signs. Plots raw and length-normalized sensor traces overlaid across volunteers, and produces panel plots covering every (sign × channel) combination. Also writes `correspondance.json` — a numeric-index → sign-name map consumed by `sankey_diagram.ipynb`.

## Configuration

In [ ]:
# Adjust these to point at your data root.
SIGNS_DIR = "SIGNS"
CORRESPONDANCE_PATH = "correspondance.json"
SIGN_INDEX = 23                        # which sign to focus on for inspection plots
FOCUS_VOLUNTEER = "Volunteer_2.csv"    # which volunteer to use for the panel plots


## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os, glob

## Discover sign directories

In [ ]:
print(os.getcwd())
route = SIGNS_DIR
signs_directory = glob.glob(os.path.join(route,"*"))
sign = SIGN_INDEX
print(signs_directory)

## Build correspondance.json

In [ ]:
dict_correspondance = {}
for indx, sign in enumerate(signs_directory):
    sign_name = sign.split("/")[-1]
    dict_correspondance[indx] = sign_name
import json

with open(CORRESPONDANCE_PATH,"w", encoding="utf-8") as handle:
    json.dump(dict_correspondance, handle)

## Load multi-volunteer CSVs for one sign

In [ ]:
#list_nam = glob.glob(os.path.join(signs_directory[sign],"*.csv"))
list_nam = list(filter(lambda x: "1" in x,glob.glob(os.path.join(signs_directory[sign],"*.csv"))))
files = list(map(lambda x: pd.read_csv(x),filter(lambda x: "1" in x,glob.glob(os.path.join(signs_directory[sign],"*.csv")))))
col_names = files[0].columns
filter_col_names = list(filter(lambda x: not("time" in x or "Unnamed" in x),list(col_names)))

## Plot raw signals

In [ ]:
fig, ax = plt.subplots(7,2,figsize=(10,40),layout="constrained")
ax = ax.flatten()
fig.suptitle(f"Seña: {signs_directory[sign]}")
for i in range(len(filter_col_names)):
    for file in range(len(files)):
        
        name = list_nam[file].split("/")[-1]
        files[file].plot(y=f"{filter_col_names[i]}",ax=ax[i],title=f"{filter_col_names[i]}",label=f"{name}")
plt.show()

## Resample to common length

In [ ]:
import numpy as np

fig, ax = plt.subplots(7,2,figsize=(10,40),layout="constrained",sharex=True)
ax = ax.flatten()
fig.suptitle(f"Seña: {signs_directory[sign]}")
for i in range(len(filter_col_names)):
    min_val = np.inf
    for file in range(len(files)):
        signal = files[file][f"{filter_col_names[i]}"]
        if len(signal) < min_val:
            min_val = len(signal)

    for file in range(len(files)):
        signal = files[file][f"{filter_col_names[i]}"]
        x_vals = np.linspace(0,min_val,len(signal))
        ax[i].plot(x_vals,signal,label=f"Volunteer {file+1}")
        ax[i].set_title(f"{filter_col_names[i]}")
        ax[i].legend()
plt.show()

## Offset to non-negative

In [ ]:
fig, ax = plt.subplots(7,2,figsize=(10,40),layout="constrained",sharex=True)
ax = ax.flatten()
fig.suptitle(f"Seña: {signs_directory[sign]}")
for i in range(len(filter_col_names)):
    min_val = np.inf
    for file in range(len(files)):
        signal = files[file][f"{filter_col_names[i]}"]
        if len(signal) < min_val:
            min_val = len(signal)

    for file in range(len(files)):
        signal = files[file][f"{filter_col_names[i]}"]
        if np.min(signal) < 0:
            signal = signal + np.abs(np.min(signal))
        x_vals = np.linspace(0,min_val,len(signal))
        ax[i].plot(x_vals,signal/np.max(np.abs(signal)),label=f"Volunteer {file+1}")
        ax[i].set_title(f"{filter_col_names[i]}")
        ax[i].legend()
plt.show()

## Color palette

In [ ]:
colors = [
    "#1f77b4",  # Azul
    "#ff7f0e",  # Naranja
    "#2ca02c",  # Verde
    "#d62728",  # Rojo
    "#9467bd",  # Morado
    "#8c564b",  # Marrón
    "#e377c2",  # Rosa
    "#7f7f7f",  # Gris
    "#bcbd22",  # Amarillo
    "#17becf",  # Cian
    "#aec7e8",  # Azul claro
    "#ffbb78",  # Naranja claro
    "#98df8a",  # Verde claro
    "#ff9896",  # Rojo claro
    "#c5b0d5",  # Morado claro
    "#c49c94",  # Marrón claro
    "#f7b6d2",  # Rosa claro
    "#c7c7c7",  # Gris claro
    "#dbdb8d",  # Amarillo claro
    "#9edae5",  # Cian claro
    "#393b79",  # Azul oscuro
    "#637939",  # Verde oscuro
    "#8c6d31",  # Marrón oscuro
    "#843c39",  # Rojo oscuro
    "#7b4173",  # Morado oscuro
    "#d6616b",  # Rosa oscuro
    "#525252",  # Gris oscuro
    "#bd9e39",  # Amarillo oscuro
    "#3182bd",  # Azul medio
    "#ad494a",  # Rojo medio
    "#756bb1",  # Morado medio
    "#8ca252",  # Verde medio
    "#e7ba52",  # Amarillo medio
    "#31a354",  # Verde esmeralda
    "#fdae6b",  # Naranja esmeralda
    "#6b6ecf",  # Azul esmeralda
    "#9c9ede",  # Morado esmeralda
    "#cedb9c",  # Verde lima
    "#de9ed6",  # Rosa lavanda
    "#b5cf6b",  # Verde limón
    "#8c6d31",  # Marrón oliva
    "#637939",  # Verde oliva
    "#e7cb94",  # Amarillo oliva
    "#843c39",  # Rojo óxido
    "#ad494a",  # Rojo salmón
    "#d6616b",  # Rosa salmón
    "#756bb1",  # Morado salmón
    "#3182bd",  # Azul turquesa
    "#6b6ecf",  # Azul violeta
    "#9c9ede",  # Morado violeta
    "#fdae6b",  # Naranja amarillo
    "#fdd0a2",  # Naranja amarillo claro
]

# Puedes acceder a los colores por índice, por ejemplo:
print(colors[0])  # Imprime el primer color (azul)



## Plot every sign × every channel (focus volunteer)

In [ ]:
fig,ax = plt.subplots(7,2,figsize=(10,35))

ax = ax.flatten()

for sign_name in range(len(signs_directory)):
    handle = pd.read_csv(os.path.join(signs_directory[sign_name],FOCUS_VOLUNTEER))
    filter_handle = handle.drop(["time","Unnamed: 0"],axis=1)
    channels = list(filter_handle.columns)
    for channel_indx in range(len(channels)):

        signal = filter_handle[channels[channel_indx]]
        if np.min(signal) < 0:
            signal = signal + np.abs(np.min(signal))
        ax[channel_indx].plot(signal/np.max(np.abs(signal)),label=f"{signs_directory[sign_name]}",c=f"{colors[sign_name]}")
        ax[channel_indx].set_title(channels[channel_indx])
plt.show()

## Plot upper-limb-only channels (focus volunteer)

In [ ]:
fig,ax = plt.subplots(3,2,figsize=(10,15))

ax = ax.flatten()

for sign_name in range(len(signs_directory)):
    handle = pd.read_csv(os.path.join(signs_directory[sign_name],FOCUS_VOLUNTEER))
    filter_handle = handle[["arm_rot_l","arm_rot_r","pro_sup_r","pro_sup_l","elbow_flex_l","elbow_flex_r"]]
    channels = list(filter_handle.columns)
    for channel_indx in range(len(channels)):

        signal = filter_handle[channels[channel_indx]]
        if np.min(signal) < 0:
            signal = signal + np.abs(np.min(signal))
        ax[channel_indx].plot(signal/np.max(np.abs(signal)),label=f"{signs_directory[sign_name]}",c=f"{colors[sign_name]}")
        ax[channel_indx].set_title(channels[channel_indx])
plt.show()